In [1]:
import os
import numpy as np
import torch as t

import importlib
import pipeline
import plots
importlib.reload(pipeline)
importlib.reload(plots)

from model import Config
from pipeline import (
    OptimizerSpec,
    Trainer,
    grid_search,
    aggregate_grid,
    print_grid_table,
    first_epoch_above,
)

t.manual_seed(0)
np.random.seed(0)

In [2]:

config = Config(
    p=113,
    d_model=128,
    d_mlp=512,
    num_heads=4,
    n_ctx=3,
    act_type='ReLU',
    frac_train=0.3,
    num_epochs=25_000,   
    seed=0,              # data shuffle seed 
)
print(f"Device : {config.device}")
print(f"p      : {config.p}")
print(f"train  : {int(config.frac_train * config.p ** 2)} pairs / {config.p ** 2}")

Device : mps
p      : 113
train  : 3830 pairs / 12769


In [3]:

#! fonction apasser en argument dans la grid_search pour construire les specs d'optimiseur à tester
def make_adamw(lr, weight_decay):
    """Build AdamW specs with Nanda's fixed betas=(0.9, 0.98)."""
    return [OptimizerSpec(
        name='adamw',
        lr=lr,
        weight_decay=weight_decay,
        extra={'betas': (0.9, 0.98)},   #! on tune que lr et wd
    )]

In [4]:
# Grille d'HPs + plan d'exécution
PARAM_GRID = {
    'lr':           [3e-4, 1e-3, 3e-3],   
    'weight_decay': [0.3, 1.0, 3.0],      
}

SEEDS_PHASE1   = [0, 1]                  
SEEDS_PHASE2   = [0, 1, 2, 3, 4]          
NUM_EPOCHS     = 25_000                
SAVE_ROOT      = 'runs/grid/adamw'

n_combos = len(PARAM_GRID['lr']) * len(PARAM_GRID['weight_decay'])
print(f"Plan grid search :")
print(f"  combinations    : {n_combos}")
print(f"  seeds Phase 1   : {SEEDS_PHASE1}  -> {n_combos * len(SEEDS_PHASE1)} runs")
print(f"  seeds Phase 2   : winner × {SEEDS_PHASE2}")
print(f"  num_epochs      : {NUM_EPOCHS}")
print(f"  save root       : {SAVE_ROOT}/")

Plan grid search :
  combinations    : 9
  seeds Phase 1   : [0, 1]  -> 18 runs
  seeds Phase 2   : winner × [0, 1, 2, 3, 4]
  num_epochs      : 25000
  save root       : runs/grid/adamw/


In [ ]:
results_p1 = grid_search(
    config,
    make_adamw,
    PARAM_GRID,
    seeds=SEEDS_PHASE1,
    save_root=SAVE_ROOT,
    num_epochs=NUM_EPOCHS,
    eval_every=50,
    fourier_every=None,
    warmup_steps=10,
    verbose_every=5_000,
    verbose_build=False,
)


=== grid_search: 9 combinations × 2 seeds = 18 runs ===
   params : ['lr', 'weight_decay']
   save   : runs/grid/adamw/

--- [1/9] lr0.0003__weight_decay0.3 ---
   will run     : [0, 1]

=== lr0.0003__weight_decay0.3 seed 0 (1/2) ===
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch     0 | train acc 0.009 | test acc 0.009
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch  5000 | train acc 1.000 | test acc 0.034
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch 10000 | train acc 1.000 | test acc 0.038
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch 15000 | train acc 1.000 | test acc 0.043
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch 20000 | train acc 1.000 | test acc 0.048
  [lr0.0003__weight_decay0.3_seed0 seed=0] epoch 25000 | train acc 1.000 | test acc 0.054
  saved to runs/grid/adamw/lr0.0003__weight_decay0.3/seed0/

=== lr0.0003__weight_decay0.3 seed 1 (2/2) ===
  [lr0.0003__weight_decay0.3_seed1 seed=1] epoch     0 | train acc 0.010 | test acc 0.009
  [lr0.0003__weight_decay0.

In [ ]:
rows_p1 = aggregate_grid(
    results_p1,
    param_keys=list(PARAM_GRID.keys()),
    acc_thresh=0.99,
    robustness_thresh=0.5,
)
print("=== Phase 1 results (sorted: valid by epoch_grok_median asc, then invalid) ===")
print_grid_table(rows_p1, param_keys=list(PARAM_GRID.keys()))

In [ ]:
best = rows_p1[0]
BEST_LR = best['lr']
BEST_WD = best['weight_decay']

print(f"Best HP from Phase 1 :")
print(f"  lr                 = {BEST_LR}")
print(f"  weight_decay       = {BEST_WD}")
print(f"  did_grok (2 seeds) = {best['did_grok_str']}")
print(f"  epoch_mem_median   = {best['epoch_mem_median']}")
print(f"  epoch_grok_median  = {best['epoch_grok_median']}")
print(f"  grok_gap_median    = {best['grok_gap_median']}")
print(f"  l2_final_median    = {best['l2_final_median']}")
print(f"  final_test_acc_med = {best['final_test_acc_med']}")